In [0]:
%sql
use catalog `e-commerce`;
use gold;

DIM TABLES


In [0]:
#dim customer
customers = spark.table("silver.customers")

dim_customer = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

dim_customer.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_customer")

In [0]:
#dim product
products = spark.table("silver.products")

dim_product = products.select(
    "product_id",
    "product_category_final"
)

dim_product.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_product")

In [0]:
#dim seller
sellers = spark.table("silver.sellers")

dim_seller = sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
)

dim_seller.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_seller")

In [0]:
#dim date
from pyspark.sql.functions import year, month, dayofmonth

orders = spark.table("silver.orders")

dim_date = orders.select("order_date").dropDuplicates()

dim_date = dim_date.withColumn("year", year("order_date")) \
                   .withColumn("month", month("order_date")) \
                   .withColumn("day", dayofmonth("order_date"))

dim_date.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_date")

In [0]:
#dim geo
geolocation = spark.table("silver.geolocation")

dim_geo = geolocation.select(
    "zip_code",
    "city",
    "state",
    "avg_lat",
    "avg_lng"
)

dim_geo.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_geography")

FACT TABLES


In [0]:
#fact orders
orders = spark.table("silver.orders")
payments = spark.table("silver.order_payments")

fact_orders = orders.join(payments, "order_id", "left")

fact_orders = fact_orders.select(
    "order_id",
    "customer_id",
    "order_date",
    "is_delivered",
    "is_late",
    "delivery_days",
    "total_payment"
)

fact_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_orders")

In [0]:
#fact order items
order_items = spark.table("silver.order_items")

fact_items = order_items.select(
    "order_id",
    "product_id",
    "seller_id",
    "price"
)

fact_items.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable("gold.fact_order_items")

In [0]:
#fact reviews
df = spark.table("silver.order_reviews")

fact_reviews = df.select(
    "order_id",
    "review_score",
)

fact_reviews.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_reviews")

In [0]:
#fact payments
df = spark.table("silver.order_payments")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_payments")